# Screener producer

Run manually from the financial_ingestion directory. The producer extracts source structure and publishes JSON. The consumer normalizes and saves it to MongoDB.


In [1]:
import os
os.environ.pop("RABBITMQ_URL", None)

In [5]:
from dataclasses import replace
from pathlib import Path
from urllib.parse import quote
from dotenv import dotenv_values
import financial_ingestion
import pika
from financial_ingestion.settings import Settings
from financial_ingestion.broker import initialize_topology
import requests
import csv
import logging
import os

env_path = Path(financial_ingestion.__file__).resolve().parents[1] / ".env"
env = dotenv_values(env_path)

print("Config file:", env_path)
print("User present:", bool(env.get("RABBITMQ_USER")))
print("Password present:", bool(env.get("RABBITMQ_PASSWORD")))

user = env.get("RABBITMQ_USER")
password = env.get("RABBITMQ_PASSWORD")
if not user or not password:
    raise ValueError("Add RABBITMQ_USER and RABBITMQ_PASSWORD to the displayed .env file")

url = (
    f"amqp://{quote(user, safe='')}:{quote(password, safe='')}"
    "@127.0.0.1:5672/%2F"
)
settings = replace(Settings.from_env(), rabbit_url=url)
CSV_PATH = Path(financial_ingestion.__file__).resolve().parents[2] / "symbols.csv"
with CSV_PATH.open(newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    if not reader.fieldnames or "symbol" not in reader.fieldnames:
        raise ValueError("symbols.csv must contain a symbol column")
    symbols = list(dict.fromkeys(row["symbol"].strip().upper() for row in reader if row["symbol"].strip()))
if not symbols:
    raise ValueError("symbols.csv contains no symbols")
parsed = pika.URLParameters(settings.rabbit_url)
print("AMQP username:", parsed.credentials.username)
print("Password encoded and decoded correctly:", parsed.credentials.password == password)

initialize_topology(settings)
print("RabbitMQ topology initialized")

Config file: C:\Repo\StockAnalysis_AI\financial_ingestion\.env
User present: True
Password present: True
AMQP username: financial_app
Password encoded and decoded correctly: True
RabbitMQ topology initialized


In [6]:
from financial_ingestion.producers import screener_produce
import uuid
import time
run_id = str(uuid.uuid4())

success, failed = 1, 0  # AARTIIND succeeded in the one-symbol test
with requests.Session() as session:
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    for symbol in symbols[1:]:
        try:
            print(f"Publishing screener for {symbol}...")
            message = screener_produce(symbol, session, settings, correlation_id=run_id)
            success += 1
            time.sleep(100)
            logging.info("Screener published %s message_id=%s", symbol, message["message_id"])
        except Exception:
            failed += 1
            logging.exception("Screener failed for %s; continuing", symbol)

print(f"Screener: published={success} failed={failed}")

Publishing screener for ABB...
Publishing screener for ABBOTINDIA...
Publishing screener for ABCAPITAL...
Publishing screener for ABFRL...
Publishing screener for ACC...
Publishing screener for ADANIENT...
Publishing screener for ADANIPORTS...
Publishing screener for ALKEM...
Publishing screener for AMBUJACEM...
Publishing screener for APOLLOHOSP...
Publishing screener for APOLLOTYRE...
Publishing screener for ASHOKLEY...
Publishing screener for ASIANPAINT...
Publishing screener for ASTRAL...
Publishing screener for ATUL...
Publishing screener for AUBANK...
Publishing screener for AUROPHARMA...
Publishing screener for AXISBANK...
Publishing screener for BAJAJ-AUTO...
Publishing screener for BAJAJFINSV...
Publishing screener for BAJFINANCE...
Publishing screener for BALKRISHIND...


ERROR:root:Screener failed for BALKRISHIND; continuing
Traceback (most recent call last):
  File "C:\Users\Varun\AppData\Local\Temp\ipykernel_35276\382982756.py", line 12, in <module>
    message = screener_produce(symbol, session, settings, correlation_id=run_id)
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\producers.py", line 11, in screener_produce
    payload,page=fetch_screener(session,symbol,settings)
                 ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\extractors.py", line 110, in fetch_screener
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Varun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: https://www.screener.in/company/BALKRISHIND/


Publishing screener for BALRAMCHIN...
Publishing screener for BANDHANBNK...
Publishing screener for BANKBARODA...
Publishing screener for BANKINDIA...
Publishing screener for BATAINDIA...
Publishing screener for BEL...
Publishing screener for BERGEPAINT...
Publishing screener for BHARATFORG...
Publishing screener for BHARTIARTL...
Publishing screener for BHEL...
Publishing screener for BIOCON...
Publishing screener for BOSCHLTD...
Publishing screener for BPCL...
Publishing screener for BRITANNIA...
Publishing screener for BSOFT...
Publishing screener for CANFINHOME...
Publishing screener for CANBK...
Publishing screener for CHAMBLFERT...
Publishing screener for CHOLAFIN...
Publishing screener for CIPLA...
Publishing screener for COALINDIA...
Publishing screener for COFORGE...
Publishing screener for COLPAL...
Publishing screener for CONCOR...
Publishing screener for COROMANDEL...
Publishing screener for CROMPTON...
Publishing screener for CUB...
Publishing screener for CUMMINSIND...
Pu

ERROR:root:Screener failed for GMRINFRA; continuing
Traceback (most recent call last):
  File "C:\Users\Varun\AppData\Local\Temp\ipykernel_35276\382982756.py", line 12, in <module>
    message = screener_produce(symbol, session, settings, correlation_id=run_id)
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\producers.py", line 11, in screener_produce
    payload,page=fetch_screener(session,symbol,settings)
                 ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\extractors.py", line 110, in fetch_screener
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Varun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: https://www.screener.in/company/GMRINFRA/


Publishing screener for GNFC...
Publishing screener for GODREJCP...
Publishing screener for GODREJPROP...
Publishing screener for GRANULES...
Publishing screener for GRASIM...
Publishing screener for GUJGASLTD...


...

Publishing screener for TATAPOWER...
Publishing screener for TCS...
Publishing screener for TECHM...
Publishing screener for TITAN...
Publishing screener for TORNTPHARM...
Publishing screener for TORNTPOWER...
Publishing screener for TRENT...
Publishing screener for TVSMOTOR...
Publishing screener for UBL...
Publishing screener for ULTRACEMCO...
Publishing screener for UPL...
Publishing screener for VDL...


ERROR:root:Screener failed for VDL; continuing
Traceback (most recent call last):
  File "C:\Users\Varun\AppData\Local\Temp\ipykernel_35276\382982756.py", line 12, in <module>
    message = screener_produce(symbol, session, settings, correlation_id=run_id)
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\producers.py", line 11, in screener_produce
    payload,page=fetch_screener(session,symbol,settings)
                 ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\extractors.py", line 110, in fetch_screener
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Varun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: https://www.screener.in/company/VDL/


Publishing screener for VOLTAS...
Publishing screener for WIPRO...
Publishing screener for ZOMATO...


ERROR:root:Screener failed for ZOMATO; continuing
Traceback (most recent call last):
  File "C:\Users\Varun\AppData\Local\Temp\ipykernel_35276\382982756.py", line 12, in <module>
    message = screener_produce(symbol, session, settings, correlation_id=run_id)
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\producers.py", line 11, in screener_produce
    payload,page=fetch_screener(session,symbol,settings)
                 ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Repo\StockAnalysis_AI\financial_ingestion\financial_ingestion\extractors.py", line 110, in fetch_screener
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Varun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: https://www.screener.in/company/ZOMATO/


Publishing screener for ZYDUSLIFE...
Publishing screener for INDOMIM...
Screener: published=174 failed=11


In [ ]:
import traceback
import requests
from financial_ingestion.producers import screener_produce

with requests.Session() as session:
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    try:
        result = screener_produce("VDL", session, settings)
        print("Published:", result["message_id"])
    except Exception:
        traceback.print_exc()

Traceback (most recent call last):
  File "C:\Users\Varun\AppData\Local\Temp\ipykernel_38784\1651857412.py", line 8, in <module>
    result = screener_produce(symbols[0], session, settings)
                              ^^^^^^^
NameError: name 'symbols' is not defined
